# 2.3. Подготовка и организация данных

Подготавливаются доступные каталоги Steam и RAWG из `raw/`, сформированные этапом 2.2. Проверяются ключевые поля, создаётся осторожная таблица соответствий игр, подготовленные данные сохраняются в Parquet и DuckDB. Исходные файлы не изменяются.

In [36]:
from pathlib import Path
import json
import math
import unicodedata
import re

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import ijson

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "notebooks").is_dir()
        and (path / "README.md").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Не удалось определить корень проекта. "
        "Запустите ноутбук внутри папки проекта."
    )

RAW = ROOT / "raw"
GAMES = ROOT / "processed" / "games"
WAREHOUSE = ROOT / "warehouse"

GAMES.mkdir(parents=True, exist_ok=True)
WAREHOUSE.mkdir(parents=True, exist_ok=True)

STEAM_SOURCE = RAW / "steam" / "fronkongames" / "games.json"
RAWG_SOURCE = RAW / "rawg" / "videogames_data.parquet"

for path in [STEAM_SOURCE, RAWG_SOURCE]:
    if not path.is_file():
        raise FileNotFoundError(f"Не найден файл из 2.2: {path}")

print("Корень проекта:", ROOT)
print("Steam:", STEAM_SOURCE.relative_to(ROOT))
print("RAWG:", RAWG_SOURCE.relative_to(ROOT))

Корень проекта: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Steam: raw\steam\fronkongames\games.json
RAWG: raw\rawg\videogames_data.parquet


## Подготовка и проверка качества

Для Steam сохраняются AppID, название, дата выпуска, цена, жанры, категории, положительные и отрицательные оценки, а также рекомендации. Для RAWG сохраняются ID, название и доступные характеристики игры.

Проверяются пропуски в ключевых полях и повторяющиеся идентификаторы и названия. Значения ID Steam и RAWG относятся к разным системам и не заменяют друг друга.

In [37]:
STEAM_OUTPUT = GAMES / "steam_games.parquet"

def to_text(value):
    if value is None:
        return None
    return str(value)

def to_float(value):
    try:
        number = pd.to_numeric(value, errors="coerce")
        return None if pd.isna(number) else float(number)
    except (TypeError, ValueError):
        return None

def to_int(value):
    number = to_float(value)
    if number is None or not math.isfinite(number):
        return None
    return int(number)

def to_json_text(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False)

steam_schema = pa.schema([
    ("AppID", pa.string()),
    ("name", pa.string()),
    ("release_date", pa.string()),
    ("price", pa.float64()),
    ("genres", pa.string()),
    ("categories", pa.string()),
    ("positive", pa.int64()),
    ("negative", pa.int64()),
    ("recommendations", pa.int64()),
])

steam_fields = [
    "name",
    "release_date",
    "price",
    "genres",
    "categories",
    "positive",
    "negative",
    "recommendations",
]

batch_size = 10_000
records = []
total_rows = 0
writer = None

try:
    with open(STEAM_SOURCE, "rb") as source:
        for app_id, game in ijson.kvitems(source, ""):
            records.append({
                "AppID": str(app_id),
                "name": to_text(game.get("name")),
                "release_date": to_text(game.get("release_date")),
                "price": to_float(game.get("price")),
                "genres": to_json_text(game.get("genres")),
                "categories": to_json_text(game.get("categories")),
                "positive": to_int(game.get("positive")),
                "negative": to_int(game.get("negative")),
                "recommendations": to_int(game.get("recommendations")),
            })

            if len(records) >= batch_size:
                table = pa.Table.from_pylist(records, schema=steam_schema)
                if writer is None:
                    writer = pq.ParquetWriter(
                        STEAM_OUTPUT,
                        steam_schema,
                        compression="zstd",
                    )
                writer.write_table(table)
                total_rows += len(records)
                records.clear()

    if records:
        table = pa.Table.from_pylist(records, schema=steam_schema)
        if writer is None:
            writer = pq.ParquetWriter(
                STEAM_OUTPUT,
                steam_schema,
                compression="zstd",
            )
        writer.write_table(table)
        total_rows += len(records)

finally:
    if writer is not None:
        writer.close()

if total_rows == 0:
    raise ValueError("В исходном Steam JSON не найдено записей.")

print("Steam: строк подготовлено =", total_rows)
print("Файл:", STEAM_OUTPUT.relative_to(ROOT))

Steam: строк подготовлено = 150466
Файл: processed\games\steam_games.parquet


In [38]:
RAWG_OUTPUT = GAMES / "rawg_games.parquet"

rawg_file = pq.ParquetFile(RAWG_SOURCE)
available_columns = set(rawg_file.schema_arrow.names)

rawg_columns = [
    "id",
    "name",
    "released",
    "rating",
    "rating_top",
    "ratings_count",
    "reviews_count",
    "metacritic",
    "playtime",
    "genres",
    "platforms",
    "developers",
    "publishers",
]

missing_required = {"id", "name"} - available_columns
if missing_required:
    raise ValueError(
        f"В RAWG Parquet отсутствуют обязательные поля: {missing_required}"
    )

rawg_columns = [
    column for column in rawg_columns
    if column in available_columns
]
rawg_schema = rawg_file.schema_arrow.select(rawg_columns)

writer = None
total_rows = 0

try:
    for batch in rawg_file.iter_batches(
        batch_size=50_000,
        columns=rawg_columns,
    ):
        if writer is None:
            writer = pq.ParquetWriter(
                RAWG_OUTPUT,
                rawg_schema,
                compression="zstd",
            )
        writer.write_batch(batch)
        total_rows += batch.num_rows

finally:
    if writer is not None:
        writer.close()

if total_rows == 0:
    raise ValueError("В исходном RAWG Parquet не найдено записей.")

print("RAWG: строк подготовлено =", total_rows)
print("Сохранённые поля:", rawg_columns)
print("Файл:", RAWG_OUTPUT.relative_to(ROOT))

AttributeError: 'pyarrow.lib.Schema' object has no attribute 'select'

## Краткий профиль данных

Сводка показывает размер подготовленных таблиц, пропуски и дубликаты в ключевых полях. Повторяющиеся названия учитываются отдельно, поскольку одно название может относиться к нескольким записям.

In [5]:
steam_keys = pd.read_parquet(
    STEAM_OUTPUT,
    columns=["AppID", "name"],
)
rawg_keys = pd.read_parquet(
    RAWG_OUTPUT,
    columns=["id", "name"],
)

quality = pd.DataFrame([
    {
        "source": "Steam",
        "rows": len(steam_keys),
        "missing_id": int(steam_keys["AppID"].isna().sum()),
        "missing_name": int(steam_keys["name"].isna().sum()),
        "duplicate_id": int(steam_keys["AppID"].duplicated().sum()),
        "names_used_more_than_once": int(
            steam_keys["name"].dropna().value_counts().gt(1).sum()
        ),
    },
    {
        "source": "RAWG",
        "rows": len(rawg_keys),
        "missing_id": int(rawg_keys["id"].isna().sum()),
        "missing_name": int(rawg_keys["name"].isna().sum()),
        "duplicate_id": int(rawg_keys["id"].duplicated().sum()),
        "names_used_more_than_once": int(
            rawg_keys["name"].dropna().value_counts().gt(1).sum()
        ),
    },
])

display(quality)

'source_group                                    relative_path       extension  size_mb\n        habr                           raw\\habr\\articles.json           .json     0.03\n        rawg             raw\\rawg\\jsonl\\videogames_data.jsonl          .jsonl  4101.47\n        rawg                     raw\\rawg\\videogames_data.csv            .csv  1907.80\n        rawg                 raw\\rawg\\videogames_data.parquet        .parquet   964.80\n       steam                 raw\\steam\\fronkongames\\games.csv            .csv   382.56\n       steam                raw\\steam\\fronkongames\\games.json           .json   973.38\n       steam        raw\\steam\\monthly_players\\steamcharts.csv            .csv    36.13\n       steam              raw\\steam\\recommendations\\games.csv            .csv     4.64\n       steam    raw\\steam\\recommendations\\games_metadata.json           .json    16.74\n       steam    raw\\steam\\recommendations\\recommendations.csv            .csv  1929.50\n    

'Empty DataFrame\nColumns: [relative_path, extension, size_mb, modified_at]\nIndex: []'

## Сопоставление Steam и RAWG

Названия нормализуются только для поиска совпадений. В итоговую таблицу входят совпадения, для которых название встречается ровно один раз в каждом каталоге. Это кандидаты на соответствие, а не подтверждённые связи; исходные ID и названия сохраняются раздельно.

In [6]:
def normalize_game_name(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value)).casefold()
    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"[_\s]+", " ", value)

    return value.strip() or None

steam_games = pd.read_parquet(
    STEAM_OUTPUT,
    columns=["AppID", "name"],
)
rawg_games = pd.read_parquet(
    RAWG_OUTPUT,
    columns=["id", "name"],
)

steam_games["normalized_name"] = (
    steam_games["name"].map(normalize_game_name)
)
rawg_games["normalized_name"] = (
    rawg_games["name"].map(normalize_game_name)
)

steam_counts = steam_games["normalized_name"].value_counts()
rawg_counts = rawg_games["normalized_name"].value_counts()

steam_unique = steam_games[
    steam_games["normalized_name"].map(steam_counts).eq(1)
].dropna(subset=["normalized_name"])

rawg_unique = rawg_games[
    rawg_games["normalized_name"].map(rawg_counts).eq(1)
].dropna(subset=["normalized_name"])

matches = steam_unique.merge(
    rawg_unique,
    on="normalized_name",
    how="inner",
    suffixes=("_steam", "_rawg"),
    validate="one_to_one",
)

matches = matches.rename(columns={
    "AppID": "steam_app_id",
    "id": "rawg_id",
    "name_steam": "steam_name",
    "name_rawg": "rawg_name",
})

matches["match_method"] = "exact_normalized_name"
matches["match_status"] = "unique_name_match_unverified"

matches = matches.sort_values(
    ["steam_app_id", "rawg_id"]
).reset_index(drop=True)

MATCHES_OUTPUT = GAMES / "steam_rawg_matches.parquet"
matches.to_parquet(
    MATCHES_OUTPUT,
    index=False,
    compression="zstd",
)

print("Однозначных совпадений-кандидатов:", len(matches))
print("Файл:", MATCHES_OUTPUT.relative_to(ROOT))
display(matches.head())

Профилировано файлов: 18

Статусы выполнения:


status
OK    18
Name: count, dtype: int64


Структура исходных файлов:


,relative_path,extension,columns,sample_rows,structure,status
0,raw\habr\articles.json,.json,None,NaN,JSON array,OK
1,raw\rawg\jsonl\videogames_data.jsonl,.jsonl,"[id, slug, name, released, background_image, r...",3.0,JSON Lines,OK
2,raw\rawg\videogames_data.csv,.csv,"[id, slug, name, released, background_image, r...",5.0,Tabular CSV,OK
3,raw\rawg\videogames_data.parquet,.parquet,"[id, slug, name, released, background_image, r...",5.0,Parquet; rows=899585; row_groups=1,OK
4,raw\steam\fronkongames\games.csv,.csv,"[AppID, Name, Release date, Estimated owners, ...",5.0,Tabular CSV,OK
5,raw\steam\fronkongames\games.json,.json,None,NaN,JSON object,OK
6,raw\steam\monthly_players\steamcharts.csv,.csv,"[month, avg_players, gain, gain_percent, peak_...",5.0,Tabular CSV,OK
7,raw\steam\recommendations\games.csv,.csv,"[app_id, title, date_release, win, mac, linux,...",5.0,Tabular CSV,OK
8,raw\steam\recommendations\games_metadata.json,.json,None,NaN,JSON object,OK
9,raw\steam\recommendations\recommendations.csv,.csv,"[app_id, helpful, funny, date, is_recommended,...",5.0,Tabular CSV,OK


## Аналитическое хранилище

Подготовленные Parquet-таблицы загружаются в DuckDB. В базе хранятся отдельные таблицы Steam и RAWG, а также таблица кандидатов на сопоставление. Исходные ID платформ не объединяются в один идентификатор.

In [7]:
import ijson
import pandas as pd

steam_json_path = RAW / "steam" / "fronkongames" / "games.json"

steam_records = []

with open(steam_json_path, "rb") as file:
    for app_id, game in ijson.kvitems(file, ""):
        steam_records.append({
            "AppID": app_id,
            **game
        })

        if len(steam_records) == 5:
            break

steam_sample = pd.DataFrame(steam_records)

print("Размерность выборки:", steam_sample.shape)
print("\nСтолбцы и типы данных:")
display(steam_sample.dtypes.to_frame("dtype"))

print("\nПервые 5 записей:")
display(steam_sample)

Размерность выборки: (5, 43)

Столбцы и типы данных:


,dtype
AppID,str
name,str
release_date,str
required_age,int64
price,object
dlc_count,int64
detailed_description,str
about_the_game,str
short_description,str
reviews,str



Первые 5 записей:


,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.0,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


### 2.1.2. Полное профилирование Steam FronkonGames

Выполняется потоковый анализ всего JSON-файла. Определяются количество записей, пропуски по атрибутам и уникальность идентификаторов приложений. Это позволяет оценить полноту и пригодность данных для последующей обработки.

In [8]:
import ijson
from collections import Counter

field_counts = Counter()
app_ids = set()

total_rows = 0
missing_app_id = 0
duplicate_app_id = 0

with open(steam_json_path, "rb") as file:
    for app_id, game in ijson.kvitems(file, ""):
        total_rows += 1

        if not app_id:
            missing_app_id += 1
        elif app_id in app_ids:
            duplicate_app_id += 1
        else:
            app_ids.add(app_id)

        for column in steam_sample.columns[1:]:
            if column not in game or game[column] is None or game[column] == "":
                field_counts[column] += 1

steam_missing = pd.DataFrame({
    "missing_count": {
        column: field_counts[column]
        for column in steam_sample.columns
    }
})

steam_missing.loc["AppID", "missing_count"] = missing_app_id
steam_missing["missing_percent"] = (
    steam_missing["missing_count"] / total_rows * 100
).round(2)

print("Количество записей:", total_rows)
print("Количество уникальных AppID:", len(app_ids))
print("Количество повторяющихся AppID:", duplicate_app_id)

display(
    steam_missing.sort_values(
        "missing_percent", ascending=False
    )
)

Количество записей: 150466
Количество уникальных AppID: 150466
Количество повторяющихся AppID: 0


,missing_count,missing_percent
score_rank,150339,99.92
metacritic_url,146181,97.15
reviews,137693,91.51
notes,121784,80.94
website,92956,61.78
support_url,86325,57.37
support_email,25297,16.81
about_the_game,8543,5.68
detailed_description,8511,5.66
short_description,8334,5.54


### 2.1.3. Анализ вложенных полей

Отдельно исследуются поля, содержащие списки, словари и текстовые значения. Проверяется структура данных и распространённость пустых коллекций. Это необходимо для выбора способов преобразования данных в табличный формат.

In [9]:
from collections import Counter

nested_fields = [
    "reviews", "supported_languages", "full_audio_languages",
    "packages", "developers", "publishers", "categories",
    "genres", "screenshots", "movies", "tags"
]

field_types = {field: Counter() for field in nested_fields}
empty_values = Counter()

with open(steam_json_path, "rb") as file:
    for _, game in ijson.kvitems(file, ""):
        for field in nested_fields:
            value = game.get(field)

            field_types[field][type(value).__name__] += 1

            if value == [] or value == {} or value == "":
                empty_values[field] += 1

nested_profile = pd.DataFrame({
    "value_types": {
        field: dict(field_types[field])
        for field in nested_fields
    },
    "empty_count": {
        field: empty_values[field]
        for field in nested_fields
    }
})

nested_profile["empty_percent"] = (
    nested_profile["empty_count"] / total_rows * 100
).round(2)

display(nested_profile)

,value_types,empty_count,empty_percent
reviews,{'str': 150466},137693,91.51
supported_languages,{'list': 150466},8402,5.58
full_audio_languages,{'list': 150466},90353,60.05
packages,{'list': 150466},28996,19.27
developers,{'list': 150466},8456,5.62
publishers,{'list': 150466},8856,5.89
categories,{'list': 150466},8981,5.97
genres,{'list': 150466},8437,5.61
screenshots,{'list': 150466},6039,4.01
movies,{'list': 150466},150466,100.00


### 2.2.1. Профилирование качества данных RAWG

На данном этапе проводится оценка полноты и уникальности записей RAWG Games Dataset. Проверяются количество записей, уникальность идентификаторов и распределение пропущенных значений по атрибутам.

Особое внимание уделяется полям, характеризующим рейтинги, жанры, платформы, разработчиков и издателей, поскольку они могут использоваться при дальнейшем анализе и интеграции игровых данных.

In [10]:
import pyarrow.parquet as pq

rawg_path = RAW / "rawg" / "videogames_data.parquet"
rawg_file = pq.ParquetFile(rawg_path)

print("Количество записей:", rawg_file.metadata.num_rows)
print("Количество столбцов:", len(rawg_file.schema.names))

print("\nСтолбцы и типы данных:")
display(pd.DataFrame({
    "column": rawg_file.schema_arrow.names,
    "dtype": [str(t) for t in rawg_file.schema_arrow.types]
}))

print("\nПервые 3 записи:")
rawg_sample = next(rawg_file.iter_batches(batch_size=3)).to_pandas()
display(rawg_sample)


Количество записей: 899585
Количество столбцов: 54

Столбцы и типы данных:


,column,dtype
0,id,int64
1,slug,string
2,name,string
3,released,string
4,background_image,string
5,rating,double
6,rating_top,double
7,ratings_count,double
8,reviews_text_count,double
9,added,double



Первые 3 записи:


,id,slug,name,released,background_image,rating,rating_top,ratings_count,reviews_text_count,added,...,screenshots_count,parent_platforms,game_series_count,description,description_raw,parents_count,reddit_name,reddit_description,reddit_logo,tba
0,3498,grand-theft-auto-v,Grand Theft Auto V,2013-09-17,https://media.rawg.io/media/games/20a/20aa03a1...,4.47,5.0,7409.0,135.0,22635.0,...,55.0,PC|PlayStation|Xbox,12.0,"Rockstar Games went bigger, since their previo...","Rockstar Games went bigger, since their previo...",NaN,nan,nan,nan,NaN
1,3328,the-witcher-3-wild-hunt,The Witcher 3: Wild Hunt,2015-05-18,https://media.rawg.io/media/games/618/618c2031...,4.64,5.0,7234.0,120.0,22285.0,...,107.0,PC|PlayStation|Xbox|Apple Macintosh|Nintendo,9.0,"The third game in a series, it holds nothing b...","The third game in a series, it holds nothing b...",NaN,nan,nan,nan,NaN
2,4200,portal-2,Portal 2,2011-04-18,https://media.rawg.io/media/games/2ba/2bac0e87...,4.58,5.0,6118.0,57.0,20926.0,...,24.0,PC|PlayStation|Xbox|Apple Macintosh|Linux,3.0,Portal 2 is a first-person puzzle game develop...,Portal 2 is a first-person puzzle game develop...,NaN,nan,nan,nan,NaN


### 2.2.2 Профилирование пропусков

In [11]:
from collections import Counter
import pyarrow.parquet as pq

rawg_file = pq.ParquetFile(rawg_path)

missing_counts = Counter()
rawg_ids = set()
total_rows = 0

for batch in rawg_file.iter_batches(batch_size=50_000):
    data = batch.to_pydict()
    total_rows += batch.num_rows

    for column, values in data.items():
        missing_counts[column] += sum(
            value is None
            or value == ""
            or (column == "added_by_status" and value == "None")
            for value in values
        )

    rawg_ids.update(data["id"])

rawg_missing = pd.DataFrame({
    "missing_count": missing_counts
})

rawg_missing["missing_percent"] = (
    rawg_missing["missing_count"] / total_rows * 100
).round(2)

rawg_missing = rawg_missing.sort_values(
    "missing_percent", ascending=False
)

print("Количество записей:", total_rows)
print("Количество уникальных id:", len(rawg_ids))
print("Количество повторяющихся id:", total_rows - len(rawg_ids))

display(rawg_missing)

Количество записей: 899585
Количество уникальных id: 899585
Количество повторяющихся id: 0


,missing_count,missing_percent
clip,899585,100.00
additions_count,897779,99.80
reddit_count,897701,99.79
tba,896846,99.70
parents_count,896219,99.63
metacritic_platforms,895232,99.52
metacritic,892455,99.21
twitch_count,889731,98.90
reviews_text_count,889180,98.84
game_series_count,888769,98.80


### 2.3.1. Профилирование данных Twitch

Twitch используется для исследования зрительского интереса к видеоиграм и активности стримеров.

Рассматриваются два набора данных: динамика игровых трансляций и просмотров, а также характеристики популярных стримеров.

Для каждого файла проверяются размерность, структура столбцов, типы данных, пропуски и полные дубликаты записей.

In [12]:
twitch_datasets = {
    "Twitch Game Data": RAW / "twitch" / "evolution" / "Twitch_game_data.csv",
    "Twitch Global Data": RAW / "twitch" / "evolution" / "Twitch_global_data.csv",
    "Twitch Streamers": RAW / "twitch" / "streamers" / "datasetV2.csv",
}

twitch_profiles = {}

for name, path in twitch_datasets.items():
    df = pd.read_csv(path, encoding="cp1252")

    twitch_profiles[name] = df

    print(f"\n{'=' * 50}")
    print(name)
    print("Количество записей:", len(df))
    print("Количество столбцов:", len(df.columns))
    print("Полных дубликатов:", df.duplicated().sum())

    print("\nСтолбцы и типы данных:")
    display(df.dtypes.to_frame("dtype"))

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print("\nПропуски:")
    if missing.empty:
        print("Пропуски отсутствуют.")
    else:
        display(missing.to_frame("missing_count"))


Twitch Game Data
Количество записей: 21000
Количество столбцов: 12
Полных дубликатов: 0

Столбцы и типы данных:


,dtype
Rank,int64
Game,str
Month,int64
Year,int64
Hours_watched,int64
Hours_streamed,int64
Peak_viewers,int64
Peak_channels,int64
Streamers,int64
Avg_viewers,int64



Пропуски:


,missing_count
Game,1



Twitch Global Data
Количество записей: 105
Количество столбцов: 9
Полных дубликатов: 0

Столбцы и типы данных:


,dtype
year,int64
Month,int64
Hours_watched,int64
Avg_viewers,int64
Peak_viewers,int64
Streams,int64
Avg_channels,int64
Games_streamed,int64
Viewer_ratio,float64



Пропуски:
Пропуски отсутствуют.

Twitch Streamers
Количество записей: 999
Количество столбцов: 17
Полных дубликатов: 0

Столбцы и типы данных:


,dtype
RANK,int64
NAME,str
LANGUAGE,str
TYPE,str
MOST_STREAMED_GAME,str
2ND_MOST_STREAMED_GAME,str
AVERAGE_STREAM_DURATION,float64
FOLLOWERS_GAINED_PER_STREAM,int64
AVG_VIEWERS_PER_STREAM,int64
AVG_GAMES_PER_STREAM,float64



Пропуски:


,missing_count
2ND_MOST_STREAMED_GAME,77


### 2.4.1 Профилирование данных YouTube

На данном этапе оценивается качество метаданных видеороликов YouTube. Проверяются структура таблицы, типы данных, пропущенные значения, полные дубликаты и уникальность идентификаторов видео.

Основное внимание уделяется идентификаторам видео, названиям игр, датам публикации и показателям пользовательской активности.

In [13]:
youtube_path = RAW / "youtube" / "youtube_videos.csv"

youtube_df = pd.read_csv(youtube_path)

print("Количество записей:", len(youtube_df))
print("Количество столбцов:", len(youtube_df.columns))
print("Полных дубликатов:", youtube_df.duplicated().sum())

print("\nСтолбцы и типы данных:")
display(youtube_df.dtypes.to_frame("dtype"))

print("\nПропуски:")
youtube_missing = youtube_df.isna().sum()
youtube_missing = youtube_missing[youtube_missing > 0].sort_values(
    ascending=False
)

if youtube_missing.empty:
    print("Пропуски отсутствуют.")
else:
    display(youtube_missing.to_frame("missing_count"))

print("\nУникальность video_id:")
print("Уникальных видео:", youtube_df["video_id"].nunique())
print("Повторяющихся video_id:", youtube_df["video_id"].duplicated().sum())

Количество записей: 298
Количество столбцов: 9
Полных дубликатов: 38

Столбцы и типы данных:


,dtype
game_name,str
video_id,str
title,str
description,str
channel_title,str
published_at,str
views,int64
likes,float64
comments,int64



Пропуски:


,missing_count
description,28
likes,4



Уникальность video_id:
Уникальных видео: 260
Повторяющихся video_id: 38


#### 2.4.2 Профилирование дополнительных JSON-источников YouTube

Дополнительно рассматриваются JSON-файлы с результатами поиска и статистикой видеороликов. Проверяется их структура и возможность использования для дополнения основной таблицы метаданных.

In [14]:
youtube_json_files = {
    "Результаты поиска": RAW / "youtube" / "youtube_search_results.json",
    "Статистика видео": RAW / "youtube" / "youtube_video_statistics.json",
}

for name, path in youtube_json_files.items():
    print(f"\n{'=' * 50}")
    print(name)

    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    print("Тип корневого объекта:", type(data).__name__)

    if isinstance(data, dict):
        print("Количество ключей:", len(data))
        print("Ключи:", list(data.keys())[:20])

    elif isinstance(data, list):
        print("Количество элементов:", len(data))

        if data:
            print("Тип первого элемента:", type(data[0]).__name__)

            if isinstance(data[0], dict):
                print("Поля первого элемента:", list(data[0].keys()))
                display(pd.DataFrame(data[:3]))
            else:
                display(data[:3])


Результаты поиска
Тип корневого объекта: list
Количество элементов: 298
Тип первого элемента: dict
Поля первого элемента: ['game_name', 'video_id', 'title', 'description', 'channel_title', 'published_at', 'views', 'likes', 'comments']


,game_name,video_id,title,description,channel_title,published_at,views,likes,comments
0,Counter-Strike 2,QVctog5F_jY,Counter Strike 2 Ranked Gameplay 4K (No Commen...,Counter Strike 2 Ranked Gameplay 4K (No Commen...,Hytron,2024-04-24T13:00:32Z,2398143,20048,448
1,Counter-Strike 2,x-m24USPutM,Counter Strike 2 Gameplay 4K (No Commentary),Counter Strike 2 Ranked Gameplay 4K (No Commen...,Hytron,2023-11-30T13:00:49Z,98688,783,16
2,Counter-Strike 2,nSE38xjMLqE,Counter-Strike 2 - Launch Trailer,Counter-Strike 2 is here! A free upgrade to CS...,Valve,2023-09-27T20:43:52Z,2771070,129960,12737



Статистика видео
Тип корневого объекта: dict
Количество ключей: 260
Ключи: ['b2Fvy2VL9Js', 'PYmnwF9L7j0', 'E3Huy2cdih0', '6Tmzc7KnKxc', 'LGDxaBJo_nQ', 'ubJOgsO4lRE', 'P4WygbqjFoI', 'nSE38xjMLqE', 'MCtJepINOnw', 'WQGisOPW8mc', 'CLYJTle8M6g', 'UnA7tepsc7s', 'j2FFa49E1rM', 'vSLivW3p3Y0', 'SB3siHVCHpY', 'N1F8ubyFdMc', 'isUnm7HcppY', 'YTZ-GZPTND8', '3NuBc7MkqVI', 'qCeWi63L1UQ']


In [15]:
stats_path = RAW / "youtube" / "youtube_video_statistics.json"

with open(stats_path, "r", encoding="utf-8") as file:
    video_stats = json.load(file)

video_id, stats = next(iter(video_stats.items()))

print("Количество видео:", len(video_stats))
print("Пример video_id:", video_id)
print("Тип значения:", type(stats).__name__)

if isinstance(stats, dict):
    print("Поля статистики:", list(stats.keys()))
    display(pd.DataFrame([stats]))
else:
    print("Пример значения:", stats)

Количество видео: 260
Пример video_id: b2Fvy2VL9Js
Тип значения: dict
Поля статистики: ['viewCount', 'likeCount', 'favoriteCount', 'commentCount']


,viewCount,likeCount,favoriteCount,commentCount
0,596506,20814,0,3090


### 2.5.1. Профилирование текстовых данных Habr

На данном этапе оценивается структура собранных публикаций Habr. Проверяются количество статей, состав атрибутов и наличие текстового содержимого.

Данные рассматриваются как неструктурированный текстовый источник, который в дальнейшем может использоваться для анализа тематик и содержательных особенностей публикаций об игровой индустрии.

In [16]:
habr_path = RAW / "habr" / "articles.json"

with open(habr_path, "r", encoding="utf-8") as file:
    habr_data = json.load(file)

print("Тип корневого объекта:", type(habr_data).__name__)
print("Количество статей:", len(habr_data) if isinstance(habr_data, list) else len(habr_data))

if isinstance(habr_data, list) and habr_data:
    print("\nПоля статьи:")
    print(list(habr_data[0].keys()))

    display(pd.DataFrame(habr_data[:3]).drop(
        columns=["text", "content", "body"], errors="ignore"
    ))

    text_fields = [
        column for column in habr_data[0]
        if column.lower() in ["text", "content", "body", "article_text"]
    ]

    print("\nПоля с текстом:", text_fields)

    if text_fields:
        field = text_fields[0]
        print("Длина текста первой статьи:", len(str(habr_data[0].get(field, ""))))

Тип корневого объекта: list
Количество статей: 2

Поля статьи:
['url', 'title', 'status_code', 'text']


,url,title,status_code
0,https://habr.com/ru/articles/995396/,"Только 12,5% игр в Steam получают пользователь...",200
1,https://habr.com/ru/articles/990554/,От ниши к мейнстриму: с 33 до 75% за 5 лет. Ка...,200



Поля с текстом: ['text']
Длина текста первой статьи: 5720


### 2.6.1. Профилирование иерархических данных Wikidata XML

На данном этапе исследуется структура XML-выгрузки Wikidata, содержащей сведения о видеоиграх. Проверяются корневой элемент, структура вложенности и состав атрибутов записей.

Источник рассматривается как иерархический набор данных, потенциально пригодный для дополнения сведений об играх и их интеграции с другими источниками.

In [17]:
import xml.etree.ElementTree as ET

wikidata_path = RAW / "xml" / "wikidata" / "wikidata_video_games.xml"

def clean_tag(tag):
    return tag.split("}")[-1] if "}" in tag else tag

variables = []
record_count = 0
sample_records = []

for event, elem in ET.iterparse(wikidata_path, events=("end",)):
    tag = clean_tag(elem.tag)

    if tag == "head":
        variables = [
            variable.attrib.get("name")
            for variable in elem
            if clean_tag(variable.tag) == "variable"
        ]

    elif tag == "result":
        record_count += 1

        if len(sample_records) < 3:
            record = {}

            for binding in elem:
                if clean_tag(binding.tag) != "binding":
                    continue

                name = binding.attrib.get("name")
                value = next(iter(binding), None)

                if name and value is not None:
                    record[name] = value.text

            sample_records.append(record)

        elem.clear()

    elif tag == "results":
        elem.clear()

print("Доступные поля:", variables)
print("Количество игровых записей:", record_count)

print("\nПримеры записей:")
display(pd.DataFrame(sample_records))

Доступные поля: ['game', 'gameLabel', 'releaseDate', 'genreLabel', 'platformLabel', 'developerLabel', 'publisherLabel']
Количество игровых записей: 1000

Примеры записей:


,game,gameLabel,releaseDate,genreLabel,platformLabel,developerLabel,publisherLabel
0,http://www.wikidata.org/entity/Q88240,720°,1986-12-01T00:00:00Z,skateboarding video game,Amstrad CPC,Tiertex Design Studios,Mindscape
1,http://www.wikidata.org/entity/Q88240,720°,1988-01-01T00:00:00Z,skateboarding video game,Amstrad CPC,Tiertex Design Studios,Mindscape
2,http://www.wikidata.org/entity/Q88240,720°,1989-12-01T00:00:00Z,skateboarding video game,Amstrad CPC,Tiertex Design Studios,Mindscape


In [18]:
wikidata_ids = set()
wikidata_names = set()

for event, elem in ET.iterparse(wikidata_path, events=("end",)):
    if clean_tag(elem.tag) == "result":
        record = {}

        for binding in elem:
            if clean_tag(binding.tag) == "binding":
                name = binding.attrib.get("name")
                value = next(iter(binding), None)

                if name and value is not None:
                    record[name] = value.text

        if "game" in record:
            wikidata_ids.add(record["game"])

        if "gameLabel" in record:
            wikidata_names.add(record["gameLabel"])

        elem.clear()

print("Количество строк:", record_count)
print("Уникальных игр по Wikidata ID:", len(wikidata_ids))
print("Уникальных названий:", len(wikidata_names))

Количество строк: 1000
Уникальных игр по Wikidata ID: 32
Уникальных названий: 32


### 2.7.1. Профилирование отзывов Steam

На данном этапе исследуется структура набора пользовательских отзывов Steam. Цель — определить доступные файлы и форматы данных, после чего оценить их пригодность для последующей текстовой обработки.

In [20]:
import csv

reviews_path = RAW / "steam" / "reviews" / "dataset.csv"

# Читаем только небольшой фрагмент файла
with open(reviews_path, "rb") as f:
    sample_bytes = f.read(65536)

# Определяем кодировку
sample_text = sample_bytes.decode("utf-8-sig")
reviews_encoding = "utf-8-sig"

# Определяем разделитель
dialect = csv.Sniffer().sniff(sample_text, delimiters=",;\t|")
reviews_sep = dialect.delimiter

# Загружаем только первые 5 записей
reviews_sample = pd.read_csv(
    reviews_path,
    sep=reviews_sep,
    encoding=reviews_encoding,
    nrows=5
)

print("Кодировка:", reviews_encoding)
print("Разделитель:", repr(reviews_sep))
print("Количество столбцов:", len(reviews_sample.columns))
print("\nСтолбцы:")
print(reviews_sample.columns.tolist())

display(reviews_sample.head())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Daniil\\Documents\\GitHub\\gaming-data-analysis\\raw\\steam\\reviews\\dataset.csv'

In [ ]:
from collections import Counter

total_rows = 0
missing_counts = Counter()
unique_app_ids = set()
score_counts = Counter()
empty_reviews = 0

for chunk in pd.read_csv(
    reviews_path,
    encoding=reviews_encoding,
    sep=reviews_sep,
    chunksize=100_000
):
    total_rows += len(chunk)

    missing_counts.update(chunk.isna().sum().to_dict())

    unique_app_ids.update(chunk["app_id"].dropna().unique())

    empty_reviews += (
        chunk["review_text"].fillna("").str.strip().eq("").sum()
    )

    score_counts.update(
        chunk["review_score"]
        .astype("string")
        .fillna("<NA>")
        .value_counts()
        .to_dict()
    )

print("Количество отзывов:", total_rows)
print("Уникальных игр (AppID):", len(unique_app_ids))
print("Пустых текстов:", empty_reviews)

print("\nПропуски:")
display(pd.DataFrame({
    "missing_count": pd.Series(missing_counts)
}).assign(
    missing_percent=lambda x: (
        x["missing_count"] / total_rows * 100
    ).round(2)
))

print("\nРаспределение оценок:")
display(
    pd.DataFrame(
        score_counts.items(),
        columns=["review_score", "count"]
    ).sort_values("review_score")
)

## 2.8.1 Правила подготовки данных

На основании результатов профилирования определены правила подготовки каждого источника. Обработка выполняется в отдельных таблицах, при этом исходные файлы сохраняются без изменений. Очистка и преобразование затрагивают только данные, предназначенные для дальнейшего анализа.

| Источник | Правила подготовки |
|---|---|
| **Steam (JSON)** | Использовать `AppID` в качестве идентификатора игры. Проверить корректность типов полей и преобразовать вложенные структуры в удобный для анализа формат. Поля с высокой долей пропусков не включать в основные аналитические таблицы без отдельного обоснования. |
| **RAWG** | Использовать `id` как идентификатор игры. Нормализовать пропущенные значения, включая строковое представление `None`. Вложенные характеристики жанров, платформ, разработчиков и издателей подготовить для последующего анализа. |
| **Twitch Game Data** | Сохранить помесячную детализацию. Проверить корректность временных и числовых показателей. Записи без названия игры не использовать при анализе игровых категорий. |
| **Twitch Global Data** | Сохранить как отдельный набор агрегированных показателей по месяцам. Не объединять напрямую с игровой статистикой без учёта различий в уровне агрегации. |
| **Twitch Streamers** | Сохранить характеристики стримеров и наиболее часто транслируемых игр. Пропуски во втором по популярности названии игры допустимы и не требуют удаления всей записи. |
| **YouTube** | Использовать `video_id` для идентификации роликов. Устранить повторяющиеся записи в подготовленной таблице. Данные из `youtube_video_statistics.json` использовать для обогащения основной таблицы по `video_id`. CSV и `youtube_search_results.json` рассматривать как дублирующие представления одного набора данных. |
| **Habr** | Сохранить тексты статей и их метаданные. Не удалять содержимое публикаций. Использовать материалы для качественного анализа тематики и демонстрации обработки текстовых данных, учитывая небольшой размер корпуса. |
| **Wikidata XML** | Использовать Wikidata ID для идентификации игр. Учитывать, что одной игре могут соответствовать несколько строк с различными датами выпуска, жанрами, платформами, разработчиками и издателями. При подготовке аналитической таблицы не считать каждую строку отдельной игрой. |

При объединении источников необходимо учитывать различия между идентификаторами Steam AppID, RAWG ID и Wikidata ID. Они не являются взаимозаменяемыми. Сопоставление игр по названиям требует нормализации и дополнительной проверки неоднозначных совпадений.

Таким образом, дальнейшая обработка должна сохранять исходную детализацию источников, учитывать уровень агрегации данных и исключать потерю информации при преобразовании.

### 2.8.2. Проектирование структуры подготовленных данных

Для дальнейшего анализа данные организуются в несколько логических групп: справочные таблицы, таблицы фактических показателей и таблицы исходных характеристик отдельных источников.

**1. Справочник игр (`dim_games`)**

Предназначен для хранения единой информации об игровых объектах после сопоставления записей из разных источников.

Основные поля:
- `game_key` — внутренний идентификатор игры;
- `game_name` — нормализованное название;
- дополнительные общие характеристики, если их значения удалось достоверно согласовать между источниками.

**2. Таблица соответствия идентификаторов (`bridge_game_sources`)**

Необходима для связи игровых объектов с записями в исходных источниках.

Основные поля:
- `game_key` — внутренний идентификатор игры;
- `source_name` — название источника;
- `source_game_id` — идентификатор игры в исходном источнике;
- `source_game_name` — исходное название;
- `match_method` — способ сопоставления;
- `match_status` — статус проверки совпадения.

**3. Таблицы игровых характеристик**

| Таблица | Содержание |
|---|---|
| `steam_games` | Характеристики игр из Steam |
| `rawg_games` | Жанры, платформы, разработчики, издатели и другие характеристики RAWG |
| `wikidata_games` | Даты выпуска, жанры, платформы, разработчики и издатели из Wikidata |


Исходные идентификаторы сохраняются в этих таблицах, а сопоставление с `game_key` выполняется через таблицу соответствий.

**4. Таблицы статистических показателей**

| Таблица | Содержание |
|---|---|
| `twitch_game_monthly` | Помесячная статистика Twitch по отдельным играм |
| `twitch_global_monthly` | Общая помесячная статистика платформы Twitch |
| `twitch_streamers` | Характеристики стримеров |
| `youtube_videos` | Метаданные видеороликов и статистика просмотров, лайков и комментариев |

**5. Текстовые данные**

| Таблица | Содержание |
|---|---|
| `steam_reviews` | Пользовательские отзывы Steam |
| `habr_articles` | Тексты и метаданные статей Habr |

Текстовые данные будут храниться отдельно от основных справочников. При необходимости для них будут подготовлены дополнительные признаки, в том числе TF-IDF-представления.

**Принципы организации:**

- Исходные данные сохраняются в `raw` без изменений.
- Подготовленные таблицы сохраняются в `processed`.
- Таблицы с разной детализацией не объединяются напрямую.
- Для сопоставления игровых объектов используется отдельная таблица соответствий.
- Неоднозначные совпадения не включаются в итоговую связь до проверки.
- Формат Parquet используется для хранения подготовленных таблиц и последующей работы с DuckDB.

Такая структура позволяет анализировать характеристики игр, динамику показателей Twitch, статистику YouTube и текстовые материалы независимо друг от друга, а также связывать их при наличии подтверждённых соответствий.

### 2.8.3. Приоритизация данных для исследования

Для сокращения избыточности и упрощения дальнейшей обработки источники разделены по их роли в исследовании.

**Основные источники** используются для изучения характеристик игровых продуктов. К ним относятся Steam, RAWG и Epic Games Store. На их основе планируется анализ жанровой структуры, платформ, дат выпуска и других доступных характеристик.

**Дополнительные источники** позволяют исследовать показатели популярности и активности игровой аудитории. К этой группе относятся наборы Twitch с помесячной статистикой игр, общей статистикой платформы и характеристиками стримеров.

**Текстовые источники** предназначены для анализа пользовательских отзывов, видеоматериалов и публикаций об игровой индустрии. Они включают Steam Reviews, YouTube и Habr.

Wikidata рассматривается как вспомогательный источник характеристик игр. Из-за ограниченного количества уникальных игровых объектов его данные не предполагается использовать как самостоятельную основу статистического анализа.

Для объединения сведений из разных источников предусматривается таблица соответствия идентификаторов. Единый справочник игр формируется только после проверки корректности сопоставления.

Таким образом, подготовка данных будет выполняться с учётом исследовательской ценности каждого источника, его полноты и возможности сопоставления с другими наборами.

## **Результаты профилирования.**

Набор Steam Reviews содержит 6 417 106 отзывов, относящихся к 9 972 уникальным играм. Основные поля представлены идентификатором игры (`app_id`), названием (`app_name`), текстом отзыва (`review_text`), оценкой (`review_score`) и количеством голосов (`review_votes`).

Пропуски в текстах отзывов составляют около 0,11% записей, а в названиях игр — 2,86%. Поле `app_id` заполнено полностью.

Оценка `review_score` принимает два значения: `1` и `-1`. Положительные оценки составляют около 81,96% отзывов, отрицательные — 18,04%.

Источник пригоден для анализа пользовательских отзывов и текстовых характеристик. При дальнейшей обработке необходимо исключить записи без содержательного текста, сохранить исходные значения оценок и использовать `app_id` для связи отзывов с каталогом игр Steam.

## 3. Очистка и преобразование данных с сохранением исходных файлов.

Подготовка отзывов выполняется потоковым способом, поскольку исходный CSV-файл содержит более 6 млн записей и занимает около 2 ГБ.

На данном этапе:
- приводятся типы идентификаторов и числовых показателей;
- удаляются лишние пробелы по краям текста;
- пустые текстовые значения преобразуются в пропуски;
- сохраняются записи с отсутствующими названиями игр, поскольку для их идентификации доступен `app_id`.

Исходный CSV-файл не изменяется. Подготовленный набор сохраняется в формате Parquet.

### 3.1.1 Подготовка reviews

In [21]:
import pyarrow as pa
import pyarrow.parquet as pq
from collections import Counter

reviews_path = RAW / "steam" / "reviews" / "dataset.csv"

reviews_output_dir = PROCESSED / "reviews"
reviews_output_dir.mkdir(parents=True, exist_ok=True)

reviews_output_path = reviews_output_dir / "steam_reviews.parquet"

if reviews_output_path.exists():
    raise FileExistsError(
        f"Файл уже существует: {reviews_output_path}. "
        "Чтобы не перезаписать результаты, проверь его отдельно."
    )

writer = None

total_rows = 0
missing_app_names = 0
empty_texts = 0
score_counts = Counter()

try:
    for chunk in pd.read_csv(
        reviews_path,
        encoding="utf-8-sig",
        chunksize=100_000,
        dtype={
            "app_id": "Int64",
            "app_name": "string",
            "review_text": "string",
            "review_score": "Int8",
            "review_votes": "Int64"
        }
    ):
        # Нормализация текстовых полей
        chunk["app_name"] = chunk["app_name"].str.strip()
        chunk["review_text"] = chunk["review_text"].str.strip()

        chunk["app_name"] = chunk["app_name"].replace("", pd.NA)
        chunk["review_text"] = chunk["review_text"].replace("", pd.NA)

        total_rows += len(chunk)
        missing_app_names += chunk["app_name"].isna().sum()
        empty_texts += chunk["review_text"].isna().sum()

        score_counts.update(
            chunk["review_score"].value_counts().to_dict()
        )

        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                reviews_output_path,
                table.schema,
                compression="zstd"
            )

        writer.write_table(table)

finally:
    if writer is not None:
        writer.close()

print("Подготовка завершена")
print("Количество сохранённых отзывов:", total_rows)
print("Пропуски app_name:", missing_app_names)
print("Пустые тексты:", empty_texts)
print("Распределение оценок:", dict(score_counts))
print(
    "Размер Parquet, МБ:",
    round(reviews_output_path.stat().st_size / 1024**2, 2)
)
print("Файл:", reviews_output_path)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Daniil\\Documents\\GitHub\\gaming-data-analysis\\raw\\steam\\reviews\\dataset.csv'

In [22]:
reviews_file = pq.ParquetFile(reviews_output_path)

print("Количество строк:", reviews_file.metadata.num_rows)
print("Количество столбцов:", len(reviews_file.schema.names))
print("Столбцы:", reviews_file.schema.names)
print("Размер файла, МБ:", round(reviews_output_path.stat().st_size / 1024**2, 2))

display(
    next(reviews_file.iter_batches(batch_size=5)).to_pandas()
)

FileNotFoundError: [WinError 2] Failed to open local file 'C:/Users/Daniil/Documents/GitHub/gaming-data-analysis/processed/reviews/steam_reviews.parquet'. Detail: [Windows error 2] The system cannot find the file specified.


### 3.1.2 Подготовка каталога игр Steam

Из исходного JSON формируется компактная таблица с характеристиками игр. 
Для обработки большого файла используется потоковое чтение через `ijson`.

В таблицу включаются идентификаторы, сведения о выпуске, цене, платформах, 
жанрах, категориях, разработчиках, издателях, пользовательских оценках 
и показателях активности.

Вложенные категориальные поля сохраняются в JSON-формате. Исходный файл 
не изменяется.

In [23]:
import json
import ijson
import pyarrow as pa
import pyarrow.parquet as pq

steam_output_dir = PROCESSED / "games"
steam_output_dir.mkdir(parents=True, exist_ok=True)

steam_output_path = steam_output_dir / "steam_games.parquet"

steam_columns = [
    "type", "name", "release_date", "required_age", "price", "dlc_count",
    "windows", "mac", "linux",
    "metacritic_score", "user_score", "achievements", "recommendations",
    "supported_languages", "developers", "publishers",
    "categories", "genres", "positive", "negative",
    "estimated_owners",
    "average_playtime_forever", "average_playtime_2weeks",
    "median_playtime_forever", "discount", "peak_ccu", "tags"
]

numeric_columns = {
    "required_age", "price", "dlc_count",
    "metacritic_score", "user_score", "achievements",
    "recommendations", "positive", "negative",
    "average_playtime_forever", "average_playtime_2weeks",
    "median_playtime_forever", "discount", "peak_ccu"
}

nested_columns = {
    "supported_languages", "developers", "publishers",
    "categories", "genres", "tags"
}

batch_size = 10_000
batch = []
writer = None
total_rows = 0

try:
    with open(steam_json_path, "rb") as file:
        for app_id, game in ijson.kvitems(file, ""):

            record = {"AppID": int(app_id)}

            for column in steam_columns:
                value = game.get(column)

                if column in numeric_columns:
                    value = pd.to_numeric(value, errors="coerce")

                    if pd.isna(value):
                        value = None

                elif column in nested_columns and value is not None:
                    value = json.dumps(value, ensure_ascii=False)

                record[column] = value

            batch.append(record)

            if len(batch) >= batch_size:
                table = pa.Table.from_pylist(batch)

                if writer is None:
                    writer = pq.ParquetWriter(
                        steam_output_path,
                        table.schema,
                        compression="zstd"
                    )

                writer.write_table(table)
                total_rows += len(batch)
                batch.clear()

        if batch:
            table = pa.Table.from_pylist(batch)

            if writer is None:
                writer = pq.ParquetWriter(
                    steam_output_path,
                    table.schema,
                    compression="zstd"
                )

            writer.write_table(table)
            total_rows += len(batch)

finally:
    if writer is not None:
        writer.close()

print("Подготовка Steam Games завершена")
print("Количество игр:", total_rows)
print("Количество полей:", len(steam_columns) + 1)
print("Размер Parquet, МБ:",
      round(steam_output_path.stat().st_size / 1024**2, 2))
print("Файл:", steam_output_path)

Подготовка Steam Games завершена
Количество игр: 150466
Количество полей: 28
Размер Parquet, МБ: 14.04
Файл: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\processed\games\steam_games.parquet


**Ограничение источника.** Исходный набор Steam не содержит атрибута `type`, позволяющего однозначно определить тип приложения. Поэтому на данном этапе невозможно автоматически отделить видеоигры от программного обеспечения и других типов приложений. Все записи сохранены без фильтрации по этому признаку. Это ограничение необходимо учитывать при интерпретации результатов сопоставления и последующего анализа.

### 3.1.3. Подготовка RAWG

In [24]:
rawg_output_path = PROCESSED / "games" / "rawg_games.parquet"

rawg_columns = [
    "id", "name", "slug", "released", "tba",
    "rating", "rating_top", "ratings_count", "reviews_count",
    "metacritic", "playtime", "added",
    "genres", "tags", "platforms",
    "developers", "publishers", "stores", "parent_platforms"
]

rawg_nested_columns = {
    "genres", "tags", "platforms",
    "developers", "publishers", "stores", "parent_platforms"
}

rawg_df = pd.read_parquet(rawg_path, columns=rawg_columns)

for column in rawg_nested_columns:
    rawg_df[column] = rawg_df[column].apply(
        lambda value: json.dumps(value, ensure_ascii=False)
        if value is not None else None
    )

rawg_df.to_parquet(
    rawg_output_path,
    index=False,
    compression="zstd"
)

print("Подготовка RAWG завершена")
print("Количество игр:", len(rawg_df))
print("Количество полей:", len(rawg_df.columns))
print("Размер Parquet, МБ:",
      round(rawg_output_path.stat().st_size / 1024**2, 2))

Подготовка RAWG завершена
Количество игр: 899585
Количество полей: 19
Размер Parquet, МБ: 46.79


### 3.1.4. Подготовка Twitch

In [25]:
twitch_output_dir = PROCESSED / "twitch"
twitch_output_dir.mkdir(parents=True, exist_ok=True)

twitch_filenames = {
    "games": "Twitch_game_data.csv",
    "global": "Twitch_global_data.csv",
    "streamers": "datasetV2.csv"
}

for dataset_name, filename in twitch_filenames.items():
    matches = list(RAW.rglob(filename))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"{filename}: найдено файлов — {len(matches)}. "
            f"Пути: {matches}"
        )

    source_path = matches[0]
    encoding = "cp1252" if dataset_name == "games" else "utf-8"

    df = pd.read_csv(source_path, encoding=encoding)

    output_path = twitch_output_dir / f"twitch_{dataset_name}.parquet"

    df.to_parquet(
        output_path,
        index=False,
        compression="zstd"
    )

    print(
        f"{dataset_name}: {len(df):,} строк, "
        f"{len(df.columns)} столбцов, "
        f"{output_path.stat().st_size / 1024**2:.2f} МБ"
    )

games: 21,000 строк, 12 столбцов, 0.59 МБ
global: 105 строк, 9 столбцов, 0.01 МБ
streamers: 999 строк, 17 столбцов, 0.05 МБ


### 3.1.5. Подготовка Youtube
Данные о найденных видео объединяются с дополнительной статистикой 
по идентификатору видео (`video_id`). Повторяющиеся записи удаляются 
по этому идентификатору. Исходные файлы не изменяются.

In [26]:
youtube_output_dir = PROCESSED / "games"
youtube_output_dir.mkdir(parents=True, exist_ok=True)

youtube_output_path = youtube_output_dir / "youtube_videos.parquet"

youtube_df = pd.read_csv(youtube_path)

youtube_df = youtube_df.drop_duplicates(
    subset="video_id",
    keep="first"
).copy()

with open(stats_path, "r", encoding="utf-8") as file:
    youtube_stats = json.load(file)

stats_df = pd.DataFrame.from_dict(
    youtube_stats,
    orient="index"
).rename_axis("video_id").reset_index()

youtube_df = youtube_df.merge(
    stats_df,
    on="video_id",
    how="left"
)

youtube_df.to_parquet(
    youtube_output_path,
    index=False,
    compression="zstd"
)

print("Подготовка YouTube завершена")
print("Количество видео:", len(youtube_df))
print("Количество полей:", len(youtube_df.columns))
print("Видео без дополнительной статистики:",
      youtube_df["viewCount"].isna().sum())
print("Размер Parquet, МБ:",
      round(youtube_output_path.stat().st_size / 1024**2, 2))

Подготовка YouTube завершена
Количество видео: 260
Количество полей: 13
Видео без дополнительной статистики: 0
Размер Parquet, МБ: 0.04


### 3.1.6. Подготовка Habr

In [27]:
habr_output_dir = PROCESSED / "games"
habr_output_dir.mkdir(parents=True, exist_ok=True)

habr_output_path = habr_output_dir / "habr_articles.parquet"

with open(habr_path, "r", encoding="utf-8") as file:
    habr_data = json.load(file)

habr_df = pd.DataFrame(habr_data)

habr_df.to_parquet(
    habr_output_path,
    index=False,
    compression="zstd"
)

print("Подготовка Habr завершена")
print("Количество статей:", len(habr_df))
print("Количество полей:", len(habr_df.columns))
print("Общий объём текста, символов:",
      habr_df["text"].fillna("").str.len().sum())
print("Размер Parquet, МБ:",
      round(habr_output_path.stat().st_size / 1024**2, 2))

Подготовка Habr завершена
Количество статей: 2
Количество полей: 4
Общий объём текста, символов: 19059
Размер Parquet, МБ: 0.01


### 3.1.7 Подготовка данных Wikidata

XML-результаты SPARQL-запроса преобразуются в табличный формат Parquet.
Сохраняются идентификатор игры в Wikidata, название, дата выпуска,
жанры, платформы, разработчики и издатели.

Повторяющиеся идентификаторы игр сохраняются, поскольку строки могут
содержать различные значения связанных характеристик.

In [28]:
import xml.etree.ElementTree as ET

wikidata_output_dir = PROCESSED / "games"
wikidata_output_dir.mkdir(parents=True, exist_ok=True)

wikidata_output_path = wikidata_output_dir / "wikidata_games.parquet"

namespace = {"ns": "http://www.w3.org/2005/sparql-results#"}

records = []

for event, elem in ET.iterparse(wikidata_path, events=("end",)):
    if elem.tag.endswith("result"):
        record = {}

        for binding in elem.findall("ns:binding", namespace):
            variable = binding.attrib["name"]
            value = binding.find("ns:uri", namespace)

            if value is None:
                value = binding.find("ns:literal", namespace)

            record[variable] = value.text if value is not None else None

        records.append(record)
        elem.clear()

wikidata_df = pd.DataFrame(records)

wikidata_df.to_parquet(
    wikidata_output_path,
    index=False,
    compression="zstd"
)

print("Подготовка Wikidata завершена")
print("Количество строк:", len(wikidata_df))
print("Уникальных игр:", wikidata_df["game"].nunique())
print("Количество полей:", len(wikidata_df.columns))
print("Размер Parquet, МБ:",
      round(wikidata_output_path.stat().st_size / 1024**2, 2))

Подготовка Wikidata завершена
Количество строк: 1000
Уникальных игр: 32
Количество полей: 7
Размер Parquet, МБ: 0.01


### 3.1.8 Подготовка данных Epic Games

In [29]:
epic_source_path = PROCESSED / "games" / "epic_games.parquet"

epic_output_path = PROCESSED / "games" / "epic_games_prepared.parquet"

epic_df = pd.read_parquet(epic_source_path)

epic_df.to_parquet(
    epic_output_path,
    index=False,
    compression="zstd"
)

print("Подготовка Epic Games завершена")
print("Количество игр:", len(epic_df))
print("Количество полей:", len(epic_df.columns))
print("Размер Parquet, МБ:",
      round(epic_output_path.stat().st_size / 1024**2, 2))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Daniil\\Documents\\GitHub\\gaming-data-analysis\\processed\\games\\epic_games.parquet'

### 3.2 сопоставление игр между источниками
Для интеграции данных из разных источников используется нормализация
названий игр. Она позволяет выявлять потенциальные совпадения при
различиях в регистре, пунктуации и форматировании.

Исходные названия сохраняются. Нормализованные значения используются
только для поиска соответствий.

In [32]:
import re
import unicodedata

def normalize_game_name(value):
    if pd.isna(value):
        return None

    value = unicodedata.normalize("NFKC", str(value))
    value = value.casefold()

    value = re.sub(r"[^\w\s]", " ", value)
    value = re.sub(r"[_\s]+", " ", value)

    return value.strip()

steam_games_path = PROCESSED / "games" / "steam_games.parquet"

steam_games = pd.read_parquet(steam_games_path)

steam_games["normalized_name"] = steam_games["name"].apply(
    normalize_game_name
)

rawg_games["normalized_name"] = rawg_games["name"].apply(
    normalize_game_name
)

#epic_games["normalized_name"] = epic_games["name"].apply(
#    normalize_game_name
#)

wikidata_games["normalized_name"] = wikidata_games["gameLabel"].apply(
    normalize_game_name
)

youtube_videos["normalized_name"] = youtube_videos["game_name"].apply(
    normalize_game_name
)

print("Нормализация завершена")

for name, df in {
    "Steam": steam_games,
    "RAWG": rawg_games,
    #"Epic Games": epic_games,
    "Wikidata": wikidata_games,
    "YouTube": youtube_videos
}.items():
    print(
        f"{name}: {df['normalized_name'].nunique():,} "
        f"уникальных нормализованных названий"
    )

NameError: name 'rawg_games' is not defined

### 3.2.1 Поиск точных совпадений

In [ ]:
source_frames = {
    "Steam": steam_games,
    "RAWG": rawg_games,
    "Wikidata": wikidata_games,
    "YouTube": youtube_videos
}

source_names = {
    name: set(df["normalized_name"].dropna())
    for name, df in source_frames.items()
}

match_results = []

source_list = list(source_names)

for i, source_a in enumerate(source_list):
    for source_b in source_list[i + 1:]:
        matches = source_names[source_a] & source_names[source_b]

        match_results.append({
            "source_1": source_a,
            "source_2": source_b,
            "exact_matches": len(matches)
        })

match_df = pd.DataFrame(match_results).sort_values(
    "exact_matches",
    ascending=False
)

display(match_df)

### 3.2.2. Проверка неоднозначности сопоставления

Перед объединением источников необходимо проверить уникальность нормализованных названий игр внутри каждого набора данных. Одно название может соответствовать нескольким записям, например разным версиям игры или отдельным приложениям.

Такие случаи не следует автоматически объединять: они требуют дополнительной проверки, чтобы избежать ошибочного сопоставления и размножения строк при объединении таблиц.

In [ ]:
# Проверяем количество повторяющихся нормализованных названий
ambiguity_results = []

for source_name, df in source_frames.items():
    counts = df["normalized_name"].dropna().value_counts()
    duplicated_names = counts[counts > 1]

    ambiguity_results.append({
        "source": source_name,
        "unique_names": df["normalized_name"].nunique(),
        "duplicated_names": len(duplicated_names),
        "rows_with_duplicated_names": int(duplicated_names.sum()),
        "max_name_frequency": int(counts.max())
    })

ambiguity_df = pd.DataFrame(ambiguity_results)

display(ambiguity_df)

### 3.2.3. Формирование кандидатов на сопоставление Steam и RAWG

Steam и RAWG имеют наибольшее пересечение по нормализованным названиям, поэтому они выбраны в качестве основы для первичного сопоставления игровых записей.

На данном этапе используются только точные совпадения нормализованных названий. Для каждого совпадения сохраняются исходные идентификаторы и названия игр.

Совпадения, при которых одному названию соответствует несколько записей хотя бы в одном источнике, выделяются как неоднозначные и не объединяются автоматически.

In [ ]:
# Формируем частоты названий в Steam и RAWG
steam_counts = steam_games["normalized_name"].value_counts()
rawg_counts = rawg_games["normalized_name"].value_counts()

# Оставляем только названия, присутствующие в обоих источниках
common_names = set(steam_counts.index) & set(rawg_counts.index)

# Формируем таблицу кандидатов
match_candidates = pd.DataFrame({
    "normalized_name": list(common_names)
})

match_candidates["steam_count"] = (
    match_candidates["normalized_name"].map(steam_counts)
)

match_candidates["rawg_count"] = (
    match_candidates["normalized_name"].map(rawg_counts)
)

# Классифицируем совпадения по однозначности
match_candidates["match_status"] = "ambiguous"

match_candidates.loc[
    (match_candidates["steam_count"] == 1) &
    (match_candidates["rawg_count"] == 1),
    "match_status"
] = "unique"

# Сводная статистика
display(
    match_candidates["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .to_frame("count")
)

print(f"Всего совпадающих названий: {len(match_candidates):,}")

### 3.2.4. Создание таблицы однозначных соответствий Steam и RAWG

На основе результатов проверки формируется таблица соответствий между играми из Steam и RAWG.

В таблицу включаются только записи с уникальными нормализованными названиями в обоих источниках. Это позволяет избежать ошибочного объединения нескольких игр с одинаковыми названиями.

Для каждой пары сохраняются идентификаторы исходных записей, оригинальные названия и нормализованное название. Неоднозначные совпадения остаются за пределами автоматического сопоставления и могут быть рассмотрены отдельно.

In [ ]:
# Оставляем только однозначные совпадения
unique_matches = match_candidates[
    match_candidates["match_status"] == "unique"
].copy()

# Получаем исходные идентификаторы и названия Steam
steam_unique = (
    steam_games[
        ["AppID", "name", "normalized_name"]
    ]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("normalized_name")
)

# Получаем исходные идентификаторы и названия RAWG
rawg_unique = (
    rawg_games[
        ["id", "name", "normalized_name"]
    ]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("normalized_name")
)

# Объединяем соответствия
steam_rawg_matches = (
    unique_matches[["normalized_name"]]
    .merge(
        steam_unique,
        on="normalized_name",
        how="inner"
    )
    .merge(
        rawg_unique,
        on="normalized_name",
        how="inner",
        suffixes=("_steam", "_rawg")
    )
)

print(f"Количество соответствий: {len(steam_rawg_matches):,}")

display(steam_rawg_matches.head(10))

#### 3.2.5. Формирование таблицы соответствий Steam и RAWG

По результатам сопоставления сформирована таблица соответствий между каталогами Steam и RAWG. В неё включены записи, для которых нормализованные названия совпадают, а в каждом из двух источников соответствующее название встречается только один раз.

Полученные соответствия рассматриваются как кандидаты на идентификацию одной игры, а не как полностью подтверждённые связи. Совпадение названий не исключает ложных соответствий, поэтому идентификаторы Steam AppID и RAWG ID сохраняются раздельно.

In [ ]:
# Формирование итоговой таблицы соответствий Steam и RAWG

steam_rawg_matches = steam_rawg_matches.copy()

steam_rawg_matches["match_method"] = "exact_normalized_name"
steam_rawg_matches["match_status"] = "unique_name_match_unverified"

steam_rawg_matches = steam_rawg_matches.sort_values(
    ["AppID", "id"]
).reset_index(drop=True)

matches_path = PROCESSED / "games" / "steam_rawg_matches.parquet"

steam_rawg_matches.to_parquet(
    matches_path,
    index=False
)

print(f"Количество соответствий: {len(steam_rawg_matches):,}")
print(f"Файл сохранён: {matches_path}")

display(steam_rawg_matches.head(10))

#### 3.2.6. Сопоставление Epic Games Store и RAWG

Для интеграции каталога Epic Games Store с уже сопоставленными данными Steam и RAWG выполняется поиск совпадений по нормализованным названиям игр.

Как и при сопоставлении Steam и RAWG, отдельно выделяются однозначные и неоднозначные совпадения. В итоговую таблицу соответствий включаются только названия, которые встречаются ровно один раз в обоих источниках. Такие связи рассматриваются как кандидаты на соответствие, а не как подтверждённая идентификация игр.

In [ ]:
# Сопоставление Epic Games Store и RAWG

epic_counts = epic_games["normalized_name"].dropna().value_counts()
rawg_counts = rawg_games["normalized_name"].dropna().value_counts()

common_names_epic_rawg = sorted(
    set(epic_counts.index) & set(rawg_counts.index)
)

epic_rawg_candidates = pd.DataFrame({
    "normalized_name": common_names_epic_rawg
})

epic_rawg_candidates["epic_count"] = (
    epic_rawg_candidates["normalized_name"].map(epic_counts)
)

epic_rawg_candidates["rawg_count"] = (
    epic_rawg_candidates["normalized_name"].map(rawg_counts)
)

epic_rawg_candidates["match_status"] = "ambiguous"

epic_rawg_candidates.loc[
    (epic_rawg_candidates["epic_count"] == 1) &
    (epic_rawg_candidates["rawg_count"] == 1),
    "match_status"
] = "unique"

display(
    epic_rawg_candidates["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .to_frame("count")
)

print(
    "Всего совпадающих названий:",
    len(epic_rawg_candidates)
)

#### 3.2.7. Формирование таблицы соответствий Epic Games Store и RAWG

На основе результатов сопоставления сформирована таблица соответствий между каталогами Epic Games Store и RAWG. В неё включены 789 однозначных совпадений, для которых нормализованное название встречается ровно один раз в каждом источнике.

Неоднозначные совпадения исключены из итоговой таблицы. Полученные связи рассматриваются как кандидаты на идентификацию игр и требуют осторожной интерпретации.

In [ ]:
# Формирование итоговой таблицы соответствий Epic и RAWG

epic_unique_matches = epic_rawg_candidates[
    epic_rawg_candidates["match_status"] == "unique"
].copy()

epic_unique = (
    epic_games[["name", "normalized_name"]]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("normalized_name")
)

rawg_unique = (
    rawg_games[["id", "name", "normalized_name"]]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("normalized_name")
)

epic_rawg_matches = (
    epic_unique_matches[["normalized_name"]]
    .merge(epic_unique, on="normalized_name", how="inner")
    .merge(
        rawg_unique,
        on="normalized_name",
        how="inner",
        suffixes=("_epic", "_rawg")
    )
)

epic_rawg_matches["match_method"] = "exact_normalized_name"
epic_rawg_matches["match_status"] = "unique_name_match_unverified"

epic_rawg_matches = (
    epic_rawg_matches
    .sort_values("normalized_name")
    .reset_index(drop=True)
)

epic_rawg_path = PROCESSED / "games" / "epic_rawg_matches.parquet"

epic_rawg_matches.to_parquet(epic_rawg_path, index=False)

print(f"Количество соответствий: {len(epic_rawg_matches):,}")
print(f"Файл сохранён: {epic_rawg_path}")

display(epic_rawg_matches.head(10))

#### 3.2.8. Формирование единой таблицы соответствий

Для интеграции каталогов Steam, RAWG и Epic Games Store используется RAWG ID в качестве промежуточного идентификатора связи. На его основе объединяются ранее сформированные таблицы соответствий Steam—RAWG и Epic—RAWG.

В итоговой таблице сохраняются исходные идентификаторы и названия игр из каждого источника. Отсутствие соответствия в одном из источников не приводит к удалению записи. Полученная таблица представляет собой перекрёстную карту идентификаторов, а не подтверждённый универсальный каталог игр.

In [ ]:
# Объединение таблиц соответствий через RAWG ID

steam_mapping = steam_rawg_matches[
    ["id", "AppID", "name_steam", "name_rawg"]
].rename(columns={
    "id": "rawg_id"
})

epic_mapping = epic_rawg_matches[
    ["id", "name_epic", "name_rawg"]
].rename(columns={
    "id": "rawg_id"
})

game_id_mapping = steam_mapping.merge(
    epic_mapping,
    on="rawg_id",
    how="outer",
    suffixes=("_steam", "_epic"),
    validate="one_to_one"
)

game_id_mapping["has_steam_match"] = (
    game_id_mapping["AppID"].notna()
)

game_id_mapping["has_epic_match"] = (
    game_id_mapping["name_epic"].notna()
)

game_id_mapping = (
    game_id_mapping
    .sort_values("rawg_id")
    .reset_index(drop=True)
)

mapping_path = PROCESSED / "games" / "game_id_mapping.parquet"

game_id_mapping.to_parquet(mapping_path, index=False)

print(f"Всего игр в таблице соответствий: {len(game_id_mapping):,}")
print(f"Есть соответствие Steam: {game_id_mapping['has_steam_match'].sum():,}")
print(f"Есть соответствие Epic: {game_id_mapping['has_epic_match'].sum():,}")
print(
    "Есть соответствия во всех трёх источниках:",
    (
        game_id_mapping["has_steam_match"] &
        game_id_mapping["has_epic_match"]
    ).sum()
)
print(f"Файл сохранён: {mapping_path}")

display(game_id_mapping.head(10))

#### 3.2.9. Сопоставление каталога с Wikidata

Для расширения характеристик игрового каталога используется набор данных Wikidata. Сопоставление выполняется по нормализованным названиям игр.

Поскольку в Wikidata одному названию могут соответствовать несколько записей, совпадения не рассматриваются как однозначные без дополнительной проверки. Для сохранения полноты исходных сведений неоднозначные связи не удаляются автоматически.

In [ ]:
# Оценка возможности сопоставления RAWG и Wikidata

wikidata_counts = (
    wikidata_games["normalized_name"]
    .dropna()
    .value_counts()
)

rawg_counts = (
    rawg_games["normalized_name"]
    .dropna()
    .value_counts()
)

common_wikidata_rawg = sorted(
    set(wikidata_counts.index) & set(rawg_counts.index)
)

wikidata_rawg_candidates = pd.DataFrame({
    "normalized_name": common_wikidata_rawg
})

wikidata_rawg_candidates["wikidata_count"] = (
    wikidata_rawg_candidates["normalized_name"].map(wikidata_counts)
)

wikidata_rawg_candidates["rawg_count"] = (
    wikidata_rawg_candidates["normalized_name"].map(rawg_counts)
)

wikidata_rawg_candidates["match_status"] = "ambiguous"

wikidata_rawg_candidates.loc[
    (wikidata_rawg_candidates["wikidata_count"] == 1) &
    (wikidata_rawg_candidates["rawg_count"] == 1),
    "match_status"
] = "unique"

display(
    wikidata_rawg_candidates["match_status"]
    .value_counts()
    .rename_axis("match_status")
    .to_frame("count")
)

print("Всего совпадающих названий:", len(wikidata_rawg_candidates))

#### 3.2.10. Формирование таблицы соответствий RAWG и Wikidata

По результатам сопоставления выделены 8 однозначных и 29 неоднозначных совпадений. Однозначные соответствия сохраняются в отдельную таблицу. Неоднозначные совпадения остаются в таблице кандидатов, поскольку автоматическое исключение повторяющихся записей Wikidata может привести к потере информации о характеристиках игр.

Таким образом, исходные связи Wikidata сохраняются без искусственного устранения многозначности.

In [ ]:
# Разделение результатов сопоставления RAWG и Wikidata

wikidata_unique_matches = wikidata_rawg_candidates[
    wikidata_rawg_candidates["match_status"] == "unique"
].copy()

wikidata_ambiguous_matches = wikidata_rawg_candidates[
    wikidata_rawg_candidates["match_status"] == "ambiguous"
].copy()

rawg_unique = (
    rawg_games[["id", "name", "normalized_name"]]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("normalized_name")
)

wikidata_unique = (
    wikidata_games[
        ["game", "gameLabel", "normalized_name"]
    ]
    .dropna(subset=["normalized_name"])
    .drop_duplicates("normalized_name")
)

rawg_wikidata_matches = (
    wikidata_unique_matches[["normalized_name"]]
    .merge(rawg_unique, on="normalized_name", how="inner")
    .merge(
        wikidata_unique,
        on="normalized_name",
        how="inner"
    )
)

rawg_wikidata_matches["match_method"] = "exact_normalized_name"
rawg_wikidata_matches["match_status"] = "unique_name_match_unverified"

rawg_wikidata_matches.to_parquet(
    PROCESSED / "games" / "rawg_wikidata_matches.parquet",
    index=False
)

wikidata_ambiguous_matches.to_parquet(
    PROCESSED / "games" / "rawg_wikidata_ambiguous.parquet",
    index=False
)

print("Однозначных соответствий:", len(rawg_wikidata_matches))
print("Неоднозначных названий:", len(wikidata_ambiguous_matches))

display(rawg_wikidata_matches.head(10))

In [ ]:
# Добавление идентификаторов Wikidata в общую таблицу

wikidata_mapping = rawg_wikidata_matches[
    ["id", "game", "gameLabel"]
].rename(columns={
    "id": "rawg_id",
    "game": "wikidata_uri",
    "gameLabel": "wikidata_name"
})

game_id_mapping = game_id_mapping.merge(
    wikidata_mapping,
    on="rawg_id",
    how="left",
    validate="one_to_one"
)

game_id_mapping.to_parquet(
    PROCESSED / "games" / "game_id_mapping.parquet",
    index=False
)

print("Всего записей:", len(game_id_mapping))
print(
    "Связано с Wikidata:",
    game_id_mapping["wikidata_uri"].notna().sum()
)

display(
    game_id_mapping[
        game_id_mapping["wikidata_uri"].notna()
    ]
)

In [ ]:
# Проверка пересечения Wikidata с общей таблицей

wikidata_rawg_ids = set(rawg_wikidata_matches["id"])
unified_rawg_ids = set(game_id_mapping["rawg_id"])

print("Всего однозначных связей RAWG—Wikidata:", len(wikidata_rawg_ids))
print("Присутствуют в общей таблице:", len(wikidata_rawg_ids & unified_rawg_ids))
print("Отсутствуют в общей таблице:", len(wikidata_rawg_ids - unified_rawg_ids))

display(
    rawg_wikidata_matches[
        ~rawg_wikidata_matches["id"].isin(unified_rawg_ids)
    ][["id", "name", "gameLabel"]]
)

**Результат интеграции Wikidata.** Из 8 однозначных соответствий RAWG—Wikidata только одна игра присутствует в общей таблице Steam—RAWG—Epic. Остальные 7 соответствий сохранены в отдельной таблице `rawg_wikidata_matches.parquet`, поскольку для них не обнаружено связей со Steam или Epic Games Store.

Таким образом, отсутствие игры в общей таблице не означает отсутствие соответствия между RAWG и Wikidata. Раздельное хранение результатов позволяет сохранить все найденные связи без потери данных.

### 3.2.11 Youtube

In [ ]:
# Проверка подготовленных данных YouTube

youtube_videos = pd.read_parquet(
    PROCESSED / "games" / "youtube_videos.parquet"
)

print("Количество видео:", len(youtube_videos))
print("Уникальных video_id:", youtube_videos["video_id"].nunique())
print("Уникальных поисковых запросов:", youtube_videos["game_name"].nunique())
print("Дубликатов video_id:", youtube_videos["video_id"].duplicated().sum())

display(youtube_videos.head())

**Особенности использования данных YouTube.** Подготовленный набор содержит 267 уникальных видеозаписей, полученных по 26 поисковым запросам, связанным с названиями игр. Идентификатор `video_id` позволяет однозначно идентифицировать видео, однако поле `game_name` отражает поисковый запрос, а не подтверждённую принадлежность видео к конкретной игре.

Поэтому данные YouTube сохраняются как отдельный источник для анализа активности видеоконтента и пользовательского взаимодействия (просмотры, лайки и комментарии). Автоматическое присоединение этих данных к каталогу игр по `game_name` не выполняется, поскольку оно может привести к ошибочным соответствиям.

## 5. Формирование TF-IDF-признаков

Для представления текстовых данных Steam Reviews используется метод TF-IDF (Term Frequency — Inverse Document Frequency). Он позволяет оценить значимость слов в тексте с учётом частоты их употребления в отдельном документе и во всём корпусе.

Чтобы получить признаки на уровне игровых объектов, отзывы предварительно группируются по идентификатору Steam (`app_id`). Тексты отзывов одной игры объединяются в единый документ. Таким образом, единицей анализа становится игра, а не отдельный отзыв.

Полученная TF-IDF-матрица может использоваться для исследования текстовых характеристик пользовательских отзывов и сравнения игр по содержанию отзывов. Она не является предиктивной ML-моделью.

Данные Habr и YouTube рассматриваются отдельно: две статьи Habr недостаточны для формирования репрезентативного текстового корпуса, а описания YouTube-видео имеют другую природу и не должны смешиваться с отзывами пользователей.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

reviews_path = PROCESSED / "reviews" / "steam_reviews.parquet"

reviews_df = pd.read_parquet(
    reviews_path,
    columns=["app_id", "review_text"]
)

# Исключение отзывов без содержательного текста
reviews_df = reviews_df.dropna(
    subset=["app_id", "review_text"]
).copy()

reviews_df["review_text"] = reviews_df["review_text"].str.strip()
reviews_df = reviews_df[
    reviews_df["review_text"].ne("")
]

# Объединение отзывов в один документ на игру
game_documents = (
    reviews_df.groupby("app_id", as_index=False)
    .agg(
        document=("review_text", " ".join),
        review_count=("review_text", "size")
    )
)

print("Количество игр в текстовом корпусе:", len(game_documents))
print("Количество отзывов:", game_documents["review_count"].sum())
print("Среднее количество отзывов на игру:",
      round(game_documents["review_count"].mean(), 2))

display(game_documents.head())

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=10_000,
    min_df=5,
    max_df=0.8,
    stop_words="english",
    ngram_range=(1, 1),
    sublinear_tf=True,
    dtype=np.float32
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    game_documents["document"]
)

print("TF-IDF построен")
print("Размер матрицы:", tfidf_matrix.shape)
print("Количество признаков:", len(tfidf_vectorizer.get_feature_names_out()))
print("Количество ненулевых значений:", tfidf_matrix.nnz)
print(
    "Плотность матрицы:",
    round(
        tfidf_matrix.nnz /
        (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100,
        4
    ),
    "%"
)